# vLLM TPS Sweep

Send concurrent requests to a running vLLM server and measure throughput.
Adjust `CONCURRENCY` and `NUM_PROMPTS` per cell, run, and read the results.

In [1]:
import asyncio
import json
import time
import uuid
from dataclasses import dataclass
from typing import Optional

import aiohttp
import numpy as np

# ── Server config ──
BASE_URL = "http://localhost:8000"
MODEL = "minimax-m25"
# MODEL = "MY_MODEL_NAME"

# ── Default sweep params (override per cell below) ──
INPUT_LEN = 1024   # target prompt tokens
OUTPUT_LEN = 1024  # max_tokens per request

In [2]:
# ── Prompt generation ──

_TOPICS = [
    "KV cache management and memory allocation in LLM inference engines",
    "distributed consensus beyond Raft: EPaxos, Hermes, and Flexible Paxos",
    "memory allocator design for latency-sensitive systems",
    "tensor and expert parallelism strategies in large model training",
    "lock-free and wait-free data structures: algorithms and pitfalls",
    "storage engine design trade-offs: B-tree vs LSM-tree",
    "speculative decoding: draft model selection and verification batching",
    "NUMA-aware scheduling and memory placement in multi-socket servers",
    "continuous batching in LLM serving: iteration-level scheduling",
    "flash attention and memory-efficient attention variants",
]

_PADDING = (
    "Modern computing systems face fundamental trade-offs between throughput, "
    "latency, energy efficiency, and fault tolerance. Every design decision "
    "carries implications across multiple dimensions that must be evaluated "
    "against specific workload characteristics, hardware capabilities, and "
    "operational constraints. The memory hierarchy of modern processors spans "
    "registers, L1/L2/L3 caches, main memory, and persistent storage, each "
    "differing by orders of magnitude in bandwidth, latency, and capacity. "
    "Algorithms and data structures must be designed with explicit awareness "
    "of these tiers to achieve competitive performance. Distributed systems "
    "are defined by partial failure: individual components may fail independently "
    "while others continue operating. Building reliable abstractions atop "
    "unreliable hardware requires explicit reasoning about failure modes. "
)

CHARS_PER_TOKEN = 3.8

def make_prompt(input_tokens: int, output_tokens: int, idx: int) -> str:
    uid = uuid.uuid4().hex[:20]
    topic = _TOPICS[idx % len(_TOPICS)]
    header = (
        f"[{uid}]\n\n"
        f"Write a thorough technical essay on: {topic}\n\n"
        f"Cover: motivation, core concepts, algorithms, implementation details, "
        f"performance characteristics, alternatives, and future directions.\n\n"
        f"Background context:\n"
    )
    suffix = "\n\nEssay:\n"
    target_chars = int(input_tokens * CHARS_PER_TOKEN)
    reserved = len(header) + len(suffix)
    pad_needed = max(0, target_chars - reserved)
    padding = (_PADDING * (pad_needed // len(_PADDING) + 1))[:pad_needed]
    return header + padding + suffix

print(f"Sample prompt length: ~{len(make_prompt(INPUT_LEN, OUTPUT_LEN, 0)) / CHARS_PER_TOKEN:.0f} tokens")

Sample prompt length: ~1024 tokens


In [3]:
# ── Request sender ──

@dataclass
class Result:
    input_tokens: int
    output_tokens: int
    latency_s: float
    ttft_s: Optional[float]
    finish_reason: str
    success: bool
    error: Optional[str] = None

async def send_one(
    session: aiohttp.ClientSession,
    prompt: str,
    max_tokens: int,
    sem: asyncio.Semaphore,
) -> Result:
    url = f"{BASE_URL}/v1/completions"
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "max_tokens": max_tokens,
        "temperature": 0,
        "stream": True,
        "stream_options": {"include_usage": True},
    }
    t0 = time.perf_counter()
    ttft = None
    out_tok = in_tok = 0
    finish_reason = "unknown"

    async with sem:
        try:
            async with session.post(url, json=payload, timeout=aiohttp.ClientTimeout(total=600)) as resp:
                resp.raise_for_status()
                async for raw in resp.content:
                    line = raw.decode(errors="replace").strip()
                    if not line.startswith("data: "):
                        continue
                    data = line[6:]
                    if data == "[DONE]":
                        break
                    try:
                        chunk = json.loads(data)
                    except json.JSONDecodeError:
                        continue
                    if chunk.get("usage"):
                        out_tok = chunk["usage"].get("completion_tokens", 0)
                        in_tok = chunk["usage"].get("prompt_tokens", 0)
                    choices = chunk.get("choices", [])
                    if choices:
                        if choices[0].get("text") and ttft is None:
                            ttft = time.perf_counter() - t0
                        if choices[0].get("finish_reason"):
                            finish_reason = choices[0]["finish_reason"]
            return Result(in_tok, out_tok, time.perf_counter() - t0, ttft, finish_reason, True)
        except Exception as e:
            return Result(0, 0, time.perf_counter() - t0, None, "error", False, str(e))

print("Request sender ready.")

Request sender ready.


In [4]:
# ── Run benchmark ──

async def run_bench(concurrency: int, num_prompts: int, num_warmup: int = 5,
                    input_len: int = INPUT_LEN, output_len: int = OUTPUT_LEN):
    sem = asyncio.Semaphore(concurrency)
    prompts = [make_prompt(input_len, output_len, i) for i in range(max(num_prompts, num_warmup))]
    connector = aiohttp.TCPConnector(limit=concurrency + 16)

    async with aiohttp.ClientSession(connector=connector) as session:
        # Warmup
        if num_warmup > 0:
            print(f"Warming up ({num_warmup} requests, concurrency={concurrency})...")
            wu_sem = asyncio.Semaphore(min(num_warmup, concurrency))
            await asyncio.gather(*[send_one(session, prompts[i], output_len, wu_sem) for i in range(num_warmup)])

        # Timed run
        print(f"Running {num_prompts} requests at concurrency={concurrency}...")
        t0 = time.perf_counter()
        results = await asyncio.gather(*[
            send_one(session, prompts[i % len(prompts)], output_len, sem)
            for i in range(num_prompts)
        ])
        elapsed = time.perf_counter() - t0

    good = [r for r in results if r.success]
    failed = [r for r in results if not r.success]
    total_out = sum(r.output_tokens for r in good)
    total_in = sum(r.input_tokens for r in good)
    lats = sorted(r.latency_s for r in good)
    ttfts = sorted(r.ttft_s for r in good if r.ttft_s is not None)

    print(f"\n{'='*60}")
    print(f"Concurrency: {concurrency}  |  Prompts: {num_prompts}  |  Elapsed: {elapsed:.1f}s")
    print(f"Successful: {len(good)}  |  Failed: {len(failed)}")
    if failed:
        for f in failed[:3]:
            print(f"  Error: {f.error}")
    print(f"\nTokens  — input: {total_in:,}  output: {total_out:,}  total: {total_in+total_out:,}")
    print(f"Output TPS: {total_out/elapsed:,.1f}")
    print(f"Total TPS:  {(total_in+total_out)/elapsed:,.1f}")
    if lats:
        print(f"\nLatency — mean: {np.mean(lats):.2f}s  p50: {np.median(lats):.2f}s  p99: {np.percentile(lats, 99):.2f}s")
    if ttfts:
        print(f"TTFT    — mean: {np.mean(ttfts):.2f}s  p50: {np.median(ttfts):.2f}s  p99: {np.percentile(ttfts, 99):.2f}s")

    # Finish reason breakdown
    reasons = {}
    for r in good:
        reasons[r.finish_reason] = reasons.get(r.finish_reason, 0) + 1
    print(f"\nFinish reasons: {reasons}")
    print(f"Avg output tokens: {total_out/len(good):.0f}" if good else "")
    print(f"{'='*60}")

    return {"concurrency": concurrency, "num_prompts": num_prompts, "elapsed": elapsed,
            "output_tps": total_out/elapsed, "total_tps": (total_in+total_out)/elapsed,
            "lat_p50": np.median(lats) if lats else 0, "lat_p99": np.percentile(lats, 99) if lats else 0,
            "ttft_p99": np.percentile(ttfts, 99) if ttfts else 0, "failed": len(failed),
            "avg_output_tokens": total_out/len(good) if good else 0, "results": results}

print("run_bench() ready. Usage: await run_bench(concurrency=N, num_prompts=M)")

run_bench() ready. Usage: await run_bench(concurrency=N, num_prompts=M)


## Quick sanity check
Send 1 request to verify the server is working and tokens look right.

In [5]:
r = await run_bench(concurrency=1, num_prompts=1, num_warmup=0)

Running 1 requests at concurrency=1...

Concurrency: 1  |  Prompts: 1  |  Elapsed: 9.3s
Successful: 1  |  Failed: 0

Tokens  — input: 0  output: 0  total: 0
Output TPS: 0.0
Total TPS:  0.0

Latency — mean: 9.33s  p50: 9.33s  p99: 9.33s

Finish reasons: {'unknown': 1}
Avg output tokens: 0


## Sweep cells
Run each cell to test that concurrency level. Edit `concurrency` and `num_prompts` as needed.

In [ ]:
r1 = await run_bench(concurrency=1, num_prompts=10, num_warmup=2)

In [ ]:
r4 = await run_bench(concurrency=4, num_prompts=20, num_warmup=4)

In [ ]:
r16 = await run_bench(concurrency=16, num_prompts=50, num_warmup=8)

In [ ]:
r64 = await run_bench(concurrency=64, num_prompts=100, num_warmup=10)

In [7]:
r256 = await run_bench(concurrency=256, num_prompts=200, num_warmup=10)

Warming up (10 requests, concurrency=256)...
Running 200 requests at concurrency=256...

Concurrency: 256  |  Prompts: 200  |  Elapsed: 25.9s
Successful: 200  |  Failed: 0

Tokens  — input: 120,999  output: 204,800  total: 325,799
Output TPS: 7,898.9
Total TPS:  12,565.7

Latency — mean: 25.76s  p50: 25.79s  p99: 25.91s
TTFT    — mean: 1.29s  p50: 1.27s  p99: 2.36s

Finish reasons: {'length': 200}
Avg output tokens: 1024


In [31]:
r512 = await run_bench(concurrency=512, num_prompts=2000, num_warmup=10)

Warming up (10 requests, concurrency=512)...
Running 2000 requests at concurrency=512...


CancelledError: 

## Summary

In [ ]:
# Collect all results that exist
all_runs = [v for name, v in sorted(locals().items()) if name.startswith('r') and isinstance(v, dict) and 'output_tps' in v]

if all_runs:
    print(f"{'Conc':>6}  {'Out TPS':>9}  {'Tot TPS':>9}  {'Lat P50':>8}  {'Lat P99':>8}  {'TTFT P99':>9}  {'AvgOut':>7}  {'Failed':>6}")
    print("-" * 72)
    for r in all_runs:
        print(f"{r['concurrency']:>6}  {r['output_tps']:>9.1f}  {r['total_tps']:>9.1f}  "
              f"{r['lat_p50']:>8.2f}  {r['lat_p99']:>8.2f}  {r['ttft_p99']:>9.2f}  "
              f"{r['avg_output_tokens']:>7.0f}  {r['failed']:>6}")
else:
    print("No results yet — run the cells above first.")